In [1]:
CATALOG = "movie_analytics"
RAW_SCHEMA = "raw"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
RAW_VOLUME = "source_files"

RAW_ROOT = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{RAW_VOLUME}"
BRONZE_NAMESPACE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER_NAMESPACE = f"{CATALOG}.{SILVER_SCHEMA}"

IMDB_FILES = (
    "name.basics",
    "title.basics",
    "title.crew",
    "title.principals",
    "title.ratings",
)
MOVIELENS_FILES = ("links", "movies", "ratings", "tags")

In [2]:
for file in IMDB_FILES:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", "\t")
        .csv(f"{RAW_ROOT}/imdb/{file}.tsv.gz")
    )

    display(df.limit(5))

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0031983,tt0050419,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0117057,tt0075213,tt0038355"
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949,1982,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918,2007,"writer,director,actor","tt0050986,tt0083922,tt0069467,tt0050976"


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


,tconst,directors,writers
0,tt0000001,nm0005690,\N
1,tt0000002,nm0721526,\N
2,tt0000003,nm0721526,nm0721526
3,tt0000004,nm0721526,\N
4,tt0000005,nm0005690,\N


,tconst,ordering,nconst,category,job,characters
0,tt0000001,1,nm1588970,self,\N,"[""Self""]"
1,tt0000001,2,nm0005690,director,\N,\N
2,tt0000001,3,nm0005690,producer,producer,\N
3,tt0000001,4,nm0374658,cinematographer,director of photography,\N
4,tt0000002,1,nm0721526,director,\N,\N


,tconst,averageRating,numVotes
0,tt0000001,5.7,2231
1,tt0000002,5.4,325
2,tt0000003,6.4,2385
3,tt0000004,5.0,202
4,tt0000005,6.2,3096


In [3]:
for file in MOVIELENS_FILES:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_ROOT}/movielens/{file}.csv")
    )

    display(df.limit(5))

,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


# Bronze

## Shared

In [4]:
from datetime import date
from pyspark.sql import functions as F

SNAPSHOT_DATE = date(2026, 9, 20)
SNAPSHOT_PREDICATE = (
    f"_source_snapshot_date = '{SNAPSHOT_DATE.isoformat()}'"
)

INGESTION_METADATA_COLUMNS = (
    "_source_file",
    "_source_dataset",
    "_ingested_at",
    "_source_snapshot_date",
)

IMDB_BRONZE_TABLES = [
    f"{BRONZE_NAMESPACE}.imdb_{name.replace('.', '_')}"
    for name in IMDB_FILES
]


def select_typed_columns(df, target_schema):
    typed_columns = [
        F.col(field.name).try_cast(field.dataType).alias(field.name)
        for field in target_schema.fields
    ]

    return df.select(
        *typed_columns,
        *INGESTION_METADATA_COLUMNS,
    )


def drop_snapshot_duplicates(df, *business_keys):
    deduplication_keys = [
        *business_keys,
        "_source_snapshot_date",
    ]

    return df.dropDuplicates(deduplication_keys)

#### MovieLens

In [5]:
base_path = f"{RAW_ROOT}/movielens"

movielens_bronze_dfs = {}

for name in MOVIELENS_FILES:
    path = f"{base_path}/{name}.csv"

    df = (
        spark.read
        .option("header", True)
        .csv(path)
        .withColumn("_source_file", F.lit(path))
        .withColumn("_source_dataset", F.lit("movielens_32m"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_snapshot_date", F.lit(SNAPSHOT_DATE))
    )

    movielens_bronze_dfs[f"movielens_{name}"] = df

#### IMDb

In [6]:
base_path = f"{RAW_ROOT}/imdb"

imdb_bronze_dfs = {}

for name in IMDB_FILES:
    path = f"{base_path}/{name}.tsv.gz"

    df = (
        spark.read
        .option("header", True)
        .option("sep", "\t")
        .csv(path)
        .withColumn("_source_file", F.lit(path))
        .withColumn("_source_dataset", F.lit("imdb"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_snapshot_date", F.lit(SNAPSHOT_DATE))
    )

    table_name = name.replace(".", "_")
    imdb_bronze_dfs[f"imdb_{table_name}"] = df

#### Persist Bronze tables

In [7]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_NAMESPACE}")

bronze_dfs = {
    **movielens_bronze_dfs,
    **imdb_bronze_dfs,
}
for table_name, bronze_df in bronze_dfs.items():
    target_table = f"{BRONZE_NAMESPACE}.{table_name}"

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", SNAPSHOT_PREDICATE)
        .saveAsTable(target_table)
    )

#### Data check

In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
)

results = []

for table_name in IMDB_BRONZE_TABLES:
    df = spark.table(table_name)
    # Exclude Bronze ingestion metadata.
    source_columns = [
        column
        for column in df.columns
        if column not in INGESTION_METADATA_COLUMNS
    ]

    # Calculate the row count and all \N counts in one scan.
    aggregations = [
        F.count(F.lit(1)).alias("_total_rows")
    ]

    aggregations.extend([
        F.sum(
            F.when(
                F.col(column).cast("string") == r"\N",
                1,
            ).otherwise(0)
        )
        .cast("long")
        .alias(column)
        for column in source_columns
    ])

    # An aggregate always produces one result row.
    counts = df.agg(*aggregations).collect()[0].asDict()

    total_rows = counts["_total_rows"]

    for column in source_columns:
        marker_count = counts[column] or 0

        marker_percentage = (
            round((marker_count / total_rows) * 100, 4)
            if total_rows > 0
            else 0.0
        )

        results.append((
            table_name,
            column,
            total_rows,
            marker_count,
            marker_percentage,
        ))

result_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("total_rows", LongType(), False),
    StructField("backslash_n_count", LongType(), False),
    StructField("backslash_n_percentage", DoubleType(), False),
])

bronze_null_marker_profile = spark.createDataFrame(
    results,
    schema=result_schema,
)

# Show every checked column.
display(
    bronze_null_marker_profile.orderBy(
        "table_name",
        "column_name",
    )
)

# Show only columns containing literal \N values.
display(
    bronze_null_marker_profile
    .filter(F.col("backslash_n_count") > 0)
    .orderBy(
        F.desc("backslash_n_count"),
        "table_name",
        "column_name",
    )
)

,table_name,column_name,total_rows,backslash_n_count,backslash_n_percentage
0,movie_analytics.bronze.imdb_name_basics,birthYear,15668421,14986500,95.6478
1,movie_analytics.bronze.imdb_name_basics,deathYear,15668421,15405054,98.3191
2,movie_analytics.bronze.imdb_name_basics,knownForTitles,15668421,1893385,12.0841
3,movie_analytics.bronze.imdb_name_basics,nconst,15668421,0,0.0000
4,movie_analytics.bronze.imdb_name_basics,primaryName,15668421,89,0.0006
5,movie_analytics.bronze.imdb_name_basics,primaryProfession,15668421,3177194,20.2777
6,movie_analytics.bronze.imdb_title_basics,endYear,12801419,12639581,98.7358
7,movie_analytics.bronze.imdb_title_basics,genres,12801419,542734,4.2396
8,movie_analytics.bronze.imdb_title_basics,isAdult,12801419,0,0.0000
9,movie_analytics.bronze.imdb_title_basics,originalTitle,12801419,0,0.0000


,table_name,column_name,total_rows,backslash_n_count,backslash_n_percentage
0,movie_analytics.bronze.imdb_title_principals,job,101830563,82199797,80.7221
1,movie_analytics.bronze.imdb_title_principals,characters,101830563,52185276,51.2472
2,movie_analytics.bronze.imdb_name_basics,deathYear,15668421,15405054,98.3191
3,movie_analytics.bronze.imdb_name_basics,birthYear,15668421,14986500,95.6478
4,movie_analytics.bronze.imdb_title_basics,endYear,12801419,12639581,98.7358
5,movie_analytics.bronze.imdb_title_basics,runtimeMinutes,12801419,8004048,62.5247
6,movie_analytics.bronze.imdb_title_crew,writers,12797249,6323105,49.4099
7,movie_analytics.bronze.imdb_title_crew,directors,12797249,5685193,44.4251
8,movie_analytics.bronze.imdb_name_basics,primaryProfession,15668421,3177194,20.2777
9,movie_analytics.bronze.imdb_name_basics,knownForTitles,15668421,1893385,12.0841


# Silver

#### Normalize IMDb null markers

In [9]:
from pyspark.sql.types import StringType

imdb_null_normalized_dfs = {}

for table_name in IMDB_BRONZE_TABLES:
    bronze_df = spark.table(table_name)
    normalized_columns = [
        (
            F.when(F.col(field.name) == r"\N", F.lit(None))
            .otherwise(F.col(field.name))
            .alias(field.name)
        )
        if (
            isinstance(field.dataType, StringType)
            and field.name not in INGESTION_METADATA_COLUMNS
        )
        else F.col(field.name)
        for field in bronze_df.schema.fields
    ]

    table_name = table_name.rsplit(".", 1)[-1]
    imdb_null_normalized_dfs[table_name] = (
        bronze_df.select(*normalized_columns)
    )

#### Apply explicit IMDb schemas

In [10]:
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

imdb_silver_schemas = {
    "imdb_name_basics": StructType([
        StructField("nconst", StringType(), False),
        StructField("primaryName", StringType(), True),
        StructField("birthYear", IntegerType(), True),
        StructField("deathYear", IntegerType(), True),
        StructField("primaryProfession", StringType(), True),
        StructField("knownForTitles", StringType(), True),
    ]),
    "imdb_title_basics": StructType([
        StructField("tconst", StringType(), False),
        StructField("titleType", StringType(), False),
        StructField("primaryTitle", StringType(), False),
        StructField("originalTitle", StringType(), False),
        StructField("isAdult", BooleanType(), False),
        StructField("startYear", IntegerType(), True),
        StructField("endYear", IntegerType(), True),
        StructField("runtimeMinutes", IntegerType(), True),
        StructField("genres", StringType(), True),
    ]),
    "imdb_title_crew": StructType([
        StructField("tconst", StringType(), False),
        StructField("directors", StringType(), True),
        StructField("writers", StringType(), True),
    ]),
    "imdb_title_principals": StructType([
        StructField("tconst", StringType(), False),
        StructField("ordering", IntegerType(), False),
        StructField("nconst", StringType(), False),
        StructField("category", StringType(), False),
        StructField("job", StringType(), True),
        StructField("characters", StringType(), True),
    ]),
    "imdb_title_ratings": StructType([
        StructField("tconst", StringType(), False),
        StructField("averageRating", DoubleType(), False),
        StructField("numVotes", LongType(), False),
    ]),
}

imdb_typed_dfs = {}

for table_name, normalized_df in imdb_null_normalized_dfs.items():
    imdb_typed_dfs[table_name] = select_typed_columns(
        normalized_df,
        imdb_silver_schemas[table_name],
    )

#### Normalize multi-value columns into bridge tables

In [11]:
from pyspark.sql.types import ArrayType, StringType


def explode_delimited_bridge(
    df,
    parent_key,
    source_column,
    value_column,
    delimiter=",",
):
    bridge_df = (
        df
        .where(F.col(source_column).isNotNull())
        .select(
            F.col(parent_key),
            F.explode(
                F.split(F.col(source_column), delimiter)
            ).alias(value_column),
            *INGESTION_METADATA_COLUMNS,
        )
        .withColumn(value_column, F.trim(F.col(value_column)))
        .filter(
            F.col(value_column).isNotNull()
            & (F.col(value_column) != "")
        )
    )

    return drop_snapshot_duplicates(
        bridge_df,
        parent_key,
        value_column,
    )


name_basics_df = imdb_typed_dfs["imdb_name_basics"]

imdb_name_df = name_basics_df.drop(
    "primaryProfession",
    "knownForTitles",
)

imdb_name_profession_df = explode_delimited_bridge(
    name_basics_df,
    parent_key="nconst",
    source_column="primaryProfession",
    value_column="profession",
)

imdb_name_known_for_title_df = explode_delimited_bridge(
    name_basics_df,
    parent_key="nconst",
    source_column="knownForTitles",
    value_column="tconst",
)

title_basics_df = imdb_typed_dfs["imdb_title_basics"]

imdb_title_df = title_basics_df.drop("genres")

imdb_title_genre_df = explode_delimited_bridge(
    title_basics_df,
    parent_key="tconst",
    source_column="genres",
    value_column="genre",
)

title_crew_df = imdb_typed_dfs["imdb_title_crew"]

director_df = (
    explode_delimited_bridge(
        title_crew_df,
        parent_key="tconst",
        source_column="directors",
        value_column="nconst",
    )
    .withColumn("crew_role", F.lit("director"))
)

writer_df = (
    explode_delimited_bridge(
        title_crew_df,
        parent_key="tconst",
        source_column="writers",
        value_column="nconst",
    )
    .withColumn("crew_role", F.lit("writer"))
)

imdb_title_crew_member_df = (
    director_df
    .unionByName(writer_df)
    .select(
        "tconst",
        "nconst",
        "crew_role",
        *INGESTION_METADATA_COLUMNS,
    )
)

title_principals_df = imdb_typed_dfs["imdb_title_principals"]
imdb_title_principal_df = title_principals_df.drop("characters")

imdb_title_principal_character_df = (
    title_principals_df
    .where(F.col("characters").isNotNull())
    .withColumn(
        "_parsed_characters",
        F.from_json(
            F.col("characters"),
            ArrayType(StringType()),
        ),
    )
    .select(
        "tconst",
        "ordering",
        F.explode("_parsed_characters").alias("character"),
        *INGESTION_METADATA_COLUMNS,
    )
    .withColumn("character", F.trim(F.col("character")))
    .filter(
        F.col("character").isNotNull()
        & (F.col("character") != "")
    )
)

imdb_title_principal_character_df = drop_snapshot_duplicates(
    imdb_title_principal_character_df,
    "tconst",
    "ordering",
    "character",
)

imdb_name_df = drop_snapshot_duplicates(imdb_name_df, "nconst")
imdb_title_df = drop_snapshot_duplicates(imdb_title_df, "tconst")
imdb_title_crew_member_df = drop_snapshot_duplicates(
    imdb_title_crew_member_df,
    "tconst",
    "nconst",
    "crew_role",
)
imdb_title_principal_df = drop_snapshot_duplicates(
    imdb_title_principal_df,
    "tconst",
    "ordering",
)
imdb_title_rating_df = drop_snapshot_duplicates(
    imdb_typed_dfs["imdb_title_ratings"],
    "tconst",
)

imdb_silver_dfs = {
    "imdb_name": imdb_name_df,
    "imdb_name_profession": imdb_name_profession_df,
    "imdb_name_known_for_title": imdb_name_known_for_title_df,
    "imdb_title": imdb_title_df,
    "imdb_title_genre": imdb_title_genre_df,
    "imdb_title_crew_member": imdb_title_crew_member_df,
    "imdb_title_principal": imdb_title_principal_df,
    "imdb_title_principal_character": (
        imdb_title_principal_character_df
    ),
    "imdb_title_rating": imdb_title_rating_df,
}

## MovieLens

Clean the four MovieLens source tables, replace source representations with canonical IMDb and UTC timestamp columns, and split movie genres into a bridge table. The unchanged source values remain available in Bronze, while cross-source enrichment is intentionally deferred to the integration and Gold layers.

In [12]:
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

movielens_silver_schemas = {
    "movielens_links": StructType([
        StructField("movieId", LongType(), False),
        StructField("imdbId", LongType(), True),
        StructField("tmdbId", LongType(), True),
    ]),
    "movielens_movies": StructType([
        StructField("movieId", LongType(), False),
        StructField("title", StringType(), False),
        StructField("genres", StringType(), True),
    ]),
    "movielens_ratings": StructType([
        StructField("userId", LongType(), False),
        StructField("movieId", LongType(), False),
        StructField("rating", DoubleType(), False),
        StructField("timestamp", LongType(), False),
    ]),
    "movielens_tags": StructType([
        StructField("userId", LongType(), False),
        StructField("movieId", LongType(), False),
        StructField("tag", StringType(), False),
        StructField("timestamp", LongType(), False),
    ]),
}

movielens_typed_dfs = {}

for table_name, target_schema in movielens_silver_schemas.items():
    bronze_df = spark.table(f"{BRONZE_NAMESPACE}.{table_name}")

    movielens_typed_dfs[table_name] = select_typed_columns(
        bronze_df,
        target_schema,
    )

movies_df = movielens_typed_dfs["movielens_movies"]
release_year_text = F.regexp_extract(
    F.col("title"),
    r"\((\d{4})\)$",
    1,
)

movielens_movie_df = (
    movies_df
    .withColumn("title", F.trim(F.col("title")))
    .withColumn(
        "releaseYear",
        F.when(release_year_text != "", release_year_text.cast("int")),
    )
    .drop("genres")
)
movielens_movie_df = drop_snapshot_duplicates(
    movielens_movie_df,
    "movieId",
)

movielens_movie_genre_df = explode_delimited_bridge(
    movies_df.where(
        F.col("genres").isNotNull()
        & (F.col("genres") != "(no genres listed)")
    ),
    parent_key="movieId",
    source_column="genres",
    value_column="genre",
    delimiter=r"\|",
)

links_df = movielens_typed_dfs["movielens_links"]
imdb_id_text = F.col("imdbId").cast("string")
padded_imdb_id = F.when(
    F.length(imdb_id_text) < 7,
    F.lpad(imdb_id_text, 7, "0"),
).otherwise(imdb_id_text)

movielens_link_df = (
    links_df
    .withColumn(
        "tconst",
        F.when(
            F.col("imdbId") > 0,
            F.concat(F.lit("tt"), padded_imdb_id),
        ),
    )
    .select(
        "movieId",
        "tconst",
        "tmdbId",
        *INGESTION_METADATA_COLUMNS,
    )
)
movielens_link_df = drop_snapshot_duplicates(
    movielens_link_df,
    "movieId",
)

movielens_rating_df = (
    movielens_typed_dfs["movielens_ratings"]
    .withColumn(
        "ratedAt",
        F.to_timestamp(F.from_unixtime(F.col("timestamp"))),
    )
    .select(
        "userId",
        "movieId",
        "rating",
        "ratedAt",
        *INGESTION_METADATA_COLUMNS,
    )
)
movielens_rating_df = drop_snapshot_duplicates(
    movielens_rating_df,
    "userId",
    "movieId",
    "ratedAt",
)

movielens_tag_df = (
    movielens_typed_dfs["movielens_tags"]
    .withColumn("tag", F.trim(F.col("tag")))
    .withColumn(
        "taggedAt",
        F.to_timestamp(F.from_unixtime(F.col("timestamp"))),
    )
    .select(
        "userId",
        "movieId",
        "tag",
        "taggedAt",
        *INGESTION_METADATA_COLUMNS,
    )
)
movielens_tag_df = drop_snapshot_duplicates(
    movielens_tag_df,
    "userId",
    "movieId",
    "tag",
    "taggedAt",
)

movielens_silver_dfs = {
    "movielens_movie": movielens_movie_df,
    "movielens_movie_genre": movielens_movie_genre_df,
    "movielens_link": movielens_link_df,
    "movielens_rating": movielens_rating_df,
    "movielens_tag": movielens_tag_df,
}

## Persist Silver tables

Write each cleaned model as a Delta table. Re-running a snapshot replaces only that snapshot, so the operation is idempotent while preserving older IMDb snapshots.

In [13]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_NAMESPACE}")

silver_dfs = {
    **imdb_silver_dfs,
    **movielens_silver_dfs,
}

for table_name, silver_df in silver_dfs.items():
    target_table = f"{SILVER_NAMESPACE}.{table_name}"
    silver_snapshot_df = silver_df.filter(
        F.col("_source_snapshot_date") == F.lit(SNAPSHOT_DATE)
    )

    (
        silver_snapshot_df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", SNAPSHOT_PREDICATE)
        .saveAsTable(target_table)
    )

## Silver quality checks

Report structural and MovieLens relationship problems without performing the cross-source MovieLens-to-IMDb integration, which belongs to the next stage.

In [14]:
movie_keys_df = movielens_movie_df.select(
    "movieId",
    "_source_snapshot_date",
).distinct()

quality_checks = [
    (
        "movielens_movie_null_id",
        movielens_movie_df.filter(F.col("movieId").isNull()).count(),
    ),
    (
        "movielens_link_invalid_imdb_id",
        links_df.filter(
            F.col("imdbId").isNotNull()
            & (F.col("imdbId") <= 0)
        ).count(),
    ),
    (
        "movielens_rating_invalid_value",
        movielens_rating_df.filter(
            F.col("rating").isNull()
            | (F.col("rating") < 0.5)
            | (F.col("rating") > 5.0)
        ).count(),
    ),
    (
        "movielens_rating_unparsed_timestamp",
        movielens_rating_df.filter(
            F.col("ratedAt").isNull()
        ).count(),
    ),
    (
        "movielens_tag_unparsed_timestamp",
        movielens_tag_df.filter(
            F.col("taggedAt").isNull()
        ).count(),
    ),
    (
        "movielens_rating_missing_movie",
        movielens_rating_df.join(
            movie_keys_df,
            ["movieId", "_source_snapshot_date"],
            "left_anti",
        ).count(),
    ),
    (
        "movielens_tag_missing_movie",
        movielens_tag_df.join(
            movie_keys_df,
            ["movieId", "_source_snapshot_date"],
            "left_anti",
        ).count(),
    ),
    (
        "movielens_link_missing_movie",
        movielens_link_df.join(
            movie_keys_df,
            ["movieId", "_source_snapshot_date"],
            "left_anti",
        ).count(),
    ),
    (
        "imdb_title_null_id",
        imdb_title_df.filter(F.col("tconst").isNull()).count(),
    ),
    (
        "imdb_name_null_id",
        imdb_name_df.filter(F.col("nconst").isNull()).count(),
    ),
]

silver_quality_summary_df = spark.createDataFrame(
    quality_checks,
    ["check_name", "failed_row_count"],
)

display(silver_quality_summary_df.orderBy("check_name"))

,check_name,failed_row_count
0,imdb_name_null_id,0
1,imdb_title_null_id,0
2,movielens_link_invalid_imdb_id,0
3,movielens_link_missing_movie,0
4,movielens_movie_null_id,0
5,movielens_rating_invalid_value,0
6,movielens_rating_missing_movie,0
7,movielens_rating_unparsed_timestamp,0
8,movielens_tag_missing_movie,0
9,movielens_tag_unparsed_timestamp,3
